# 02. Grid Search

Hyperparameter + blend-weight search. Loads processed features written by
`01_data_exploration.ipynb` (does **not** re-run `build_patient_features`).

Runs the per-endpoint blend-weight search (`search_blend_weights`, see
`liverrisk/blend.py`) and writes the winning weights back into
`liverrisk/best_config.json` via `config.update_config(...)`.

XGB hyperparameters and the Coxnet alpha-search settings are exposed via
`liverrisk/config.py` (`xgb_hyperparams`, `coxnet_alpha_search`) but this
notebook does **not** currently grid-search them -- the original notebook
never did either, only blend weights were tuned. The config keys exist so
that a future hyperparameter search has somewhere to write its result
without a code change; for now they just hold the original hardcoded
values (see `config.DEFAULTS`).

In [1]:
import sys
from pathlib import Path


def _find_repo_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "liverrisk" / "features.py").exists():
            return p
    raise RuntimeError("Could not locate repo root (liverrisk/features.py not found)")


REPO_ROOT = _find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print("REPO_ROOT:", REPO_ROOT)

REPO_ROOT: c:\Users\paabl\OneDrive\Documents\mlc-sp26-Liver1


In [2]:
import numpy as np
import pandas as pd

from liverrisk import config
from liverrisk.blend import search_blend_weights
from liverrisk.cv import cv_cindex_blend, cv_cindex_sksurv, cv_cindex_xgb
from liverrisk.features import load_features
from liverrisk.models import HAS_XGB, make_coxnet_pipeline, make_rsf_pipeline

PROCESSED_DIR = REPO_ROOT / "liverrisk" / "data" / "processed"

# Reduce these for a fast verification run; bump back up for a trustworthy
# final search. n_points controls the blend-weight grid density (n_points=6
# matches the original notebook); n_repeats_report controls how many times
# the OLD-vs-NEW comparison at the bottom is repeated.
N_POINTS = 6
N_REPEATS_REPORT = 3

X_hep, y_hep, hep_event, hep_time = load_features("hep", PROCESSED_DIR)
X_death, y_death, death_event, death_time = load_features("death", PROCESSED_DIR)

print(f"hepatic: n={len(X_hep)}, events={hep_event.sum()} ({hep_event.mean():.3%})")
print(f"death  : n={len(X_death)}, events={death_event.sum()} ({death_event.mean():.3%})")

hepatic: n=1253, events=47 (3.751%)
death  : n=984, events=76 (7.724%)


## Individual model CV

Informational only (not used to pick anything below) -- `n_repeats=1` keeps this affordable.

In [3]:
hep_cox_cv = cv_cindex_sksurv(make_coxnet_pipeline, X_hep, y_hep, hep_event, n_repeats=1)
death_cox_cv = cv_cindex_sksurv(make_coxnet_pipeline, X_death, y_death, death_event, n_repeats=1)
print(f"Coxnet hepatic: mean={hep_cox_cv[0]:.4f}, std={hep_cox_cv[1]:.4f}")
print(f"Coxnet death  : mean={death_cox_cv[0]:.4f}, std={death_cox_cv[1]:.4f}")

rsf_cv_factory = lambda X_fold: make_rsf_pipeline(X_fold, n_estimators=150)
hep_rsf_cv = cv_cindex_sksurv(rsf_cv_factory, X_hep, y_hep, hep_event, n_repeats=1)
death_rsf_cv = cv_cindex_sksurv(rsf_cv_factory, X_death, y_death, death_event, n_repeats=1)
print(f"RSF hepatic   : mean={hep_rsf_cv[0]:.4f}, std={hep_rsf_cv[1]:.4f}")
print(f"RSF death     : mean={death_rsf_cv[0]:.4f}, std={death_rsf_cv[1]:.4f}")

if HAS_XGB:
    hep_xgb_cv = cv_cindex_xgb(X_hep, hep_event, hep_time, n_repeats=1)
    death_xgb_cv = cv_cindex_xgb(X_death, death_event, death_time, n_repeats=1)
    print(f"XGB hepatic   : mean={hep_xgb_cv[0]:.4f}, std={hep_xgb_cv[1]:.4f}")
    print(f"XGB death     : mean={death_xgb_cv[0]:.4f}, std={death_xgb_cv[1]:.4f}")
else:
    hep_xgb_cv = death_xgb_cv = (np.nan, np.nan)

Coxnet hepatic: mean=0.7197, std=0.1058
Coxnet death  : mean=0.9373, std=0.0173
RSF hepatic   : mean=0.7998, std=0.1042
RSF death     : mean=0.9449, std=0.0146
XGB hepatic   : mean=0.7279, std=0.0816
XGB death     : mean=0.9291, std=0.0270


## Blend weight search (per endpoint)

`search_blend_weights` grid-searches `(w_cox, w_rsf, w_xgb)` triples that sum to 1, including edge cases where one or two weights are 0.

In [4]:
print("Searching blend weights (hepatic)...")
hep_weights, hep_search_mean, hep_search_df = search_blend_weights(X_hep, y_hep, hep_event, hep_time, n_points=N_POINTS)
print(f"Best hepatic weights (w_cox, w_rsf, w_xgb) = {hep_weights}, search-CV mean (1 repeat) = {hep_search_mean:.4f}")

print("\nSearching blend weights (death)...")
death_weights, death_search_mean, death_search_df = search_blend_weights(X_death, y_death, death_event, death_time, n_points=N_POINTS)
print(f"Best death weights (w_cox, w_rsf, w_xgb) = {death_weights}, search-CV mean (1 repeat) = {death_search_mean:.4f}")

Searching blend weights (hepatic)...
Best hepatic weights (w_cox, w_rsf, w_xgb) = (0.0, 1.0, 0.0), search-CV mean (1 repeat) = 0.7998

Searching blend weights (death)...
Best death weights (w_cox, w_rsf, w_xgb) = (0.4, 0.4, 0.2), search-CV mean (1 repeat) = 0.9582


In [5]:
hep_search_df.head(10)

,w_cox,w_rsf,w_xgb,mean,std
0,0.0,1.0,0.0,0.799810,0.104174
1,0.2,0.8,0.0,0.798936,0.093203
2,0.4,0.6,0.0,0.793418,0.081944
3,0.0,0.8,0.2,0.791374,0.097087
4,0.2,0.6,0.2,0.790760,0.087867
5,0.0,0.6,0.4,0.779952,0.094712
6,0.4,0.4,0.2,0.777311,0.079023
7,0.6,0.4,0.0,0.776820,0.082331
8,0.2,0.4,0.4,0.776174,0.083057
9,0.0,0.4,0.6,0.763350,0.091816


In [6]:
death_search_df.head(10)

,w_cox,w_rsf,w_xgb,mean,std
0,0.4,0.4,0.2,0.958229,0.009999
1,0.4,0.6,0.0,0.957298,0.007443
2,0.2,0.6,0.2,0.956366,0.005498
3,0.6,0.4,0.0,0.955975,0.009779
4,0.2,0.8,0.0,0.955714,0.007238
5,0.6,0.2,0.2,0.955150,0.011280
6,0.4,0.2,0.4,0.952894,0.015028
7,0.2,0.4,0.4,0.951466,0.010464
8,0.8,0.2,0.0,0.949956,0.013693
9,0.6,0.0,0.4,0.949874,0.013400


## Write the winning weights to `best_config.json`

This is the **only** place blend weights get persisted -- `config.update_config` merges these two keys into the JSON file without touching `xgb_hyperparams` / `coxnet_alpha_search`.

In [7]:
config.update_config(
    blend_weights_hep=list(hep_weights),
    blend_weights_death=list(death_weights),
)
print("Updated liverrisk/best_config.json:")
print(config.get_config())

Updated liverrisk/best_config.json:
{'xgb_hyperparams': {'n_estimators': 600, 'learning_rate': 0.025, 'max_depth': 2, 'min_child_weight': 10, 'subsample': 0.85, 'colsample_bytree': 0.85, 'reg_lambda': 5.0, 'reg_alpha': 0.5}, 'coxnet_alpha_search': {'n_alphas': 30, 'n_splits': 3}, 'blend_weights_hep': [0.0, 1.0, 0.0], 'blend_weights_death': [0.4, 0.4, 0.2]}
